Experiment Plan
1. Pilot experiment: Test 10 ultrasound images using Gemini 3.5 Flash-Lite.
2. Annotation quality assessment: Evaluate the validity of JSON outputs, image descriptions, and bounding boxes.
3. Main evaluation: Process 61 validation images using the free API tier.
4. Optional model comparison: Repeat the experiment with Gemini 3.1 Flash-Lite if API rate limits allow.
5. Performance evaluation: Compare the generated annotations against BiomedParse pseudo-labels and analyze potential errors and limitations.

In [ ]:
%pip install -q google-genai "google-auth==2.49.0"

Install the official Google Gen AI SDK for Gemini inference.
Install Gemini SDK with the Google Colab-compatible authentication version

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/ovarian_ultrasound_vlm")
MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

assert len(manifest) == 301

pilot = manifest[
    manifest["is_pilot"].astype(str).str.lower() == "true"
].copy()

assert len(pilot) == 10

print("Total images:", len(manifest))
print("Pilot images:", len(pilot))
print("Pilot case IDs:", pilot["case_id"].tolist())

Mount Google Drive and locate the previously generated image manifest

In [ ]:
from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY is missing.")

client = genai.Client(api_key=API_KEY)

MODEL_ID = "gemini-3.5-flash-lite"

print("Gemini client initialized.")
print("Selected model:", MODEL_ID)

Initialize Gemini using a free-tier model

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

class FollicleRegion(BaseModel):
    bbox: List[int] = Field(
        description="Bounding box [xmin, ymin, xmax, ymax], normalized to 0-1000."
    )
    shape: Literal["round", "oval", "irregular", "uncertain"]
    appearance: str

class UltrasoundAnnotation(BaseModel):
    follicle_like_count: int
    regions: List[FollicleRegion]
    image_quality: Literal["good", "moderate", "poor"]
    uncertain: bool
    description: str

print("Annotation schema defined successfully.")

Define the structured annotation schema for ovarian ultrasound images

In [ ]:
ANNOTATION_PROMPT = """
Analyze this 2D ovarian ultrasound image for a research experiment.

Your task is to identify and describe visually apparent
follicle-like structures.

Instructions:
1. Identify rounded or oval hypoechoic or anechoic regions
   that may correspond to ovarian follicles.
2. Estimate the number of visible follicle-like regions.
3. Provide a bounding box for each identified region.
4. Use normalized coordinates from 0 to 1000:
   [xmin, ymin, xmax, ymax].
5. Describe the visible shape and appearance.
6. Assess image quality as good, moderate, or poor.
7. Indicate uncertainty when appropriate.
8. If no structures can be identified reliably,
   return an empty regions list and count 0.
9. Do not infer medical diagnoses or patient characteristics.
10. Do not assume that every dark region is a follicle.
11. Ensure that follicle_like_count equals the number of regions.

Return structured annotations according to the provided schema.
"""

print("Annotation prompt ready.")

Define the research prompt for structured follicle-like region annotation

In [ ]:
from google.genai import types
from PIL import Image
import json

case = pilot.iloc[0]

image_path = Path(case["image_path"])

print("Case ID:", case["case_id"])
print("Image path:", image_path)

image = Image.open(image_path).convert("RGB")

display(image)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[image, ANNOTATION_PROMPT],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=UltrasoundAnnotation,
        temperature=0
    )
)

annotation = UltrasoundAnnotation.model_validate_json(
    response.text
)

# Validate annotation coordinates and count.
assert annotation.follicle_like_count == len(annotation.regions)

for region in annotation.regions:
    assert len(region.bbox) == 4
    x1, y1, x2, y2 = region.bbox
    assert 0 <= x1 < x2 <= 1000
    assert 0 <= y1 < y2 <= 1000

print("\nGemini annotation:")
print(json.dumps(annotation.model_dump(), indent=2))

Run the first Gemini annotation test on one pilot ultrasound image

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))

ax.imshow(image)

width, height = image.size

for region in annotation.regions:
    x1, y1, x2, y2 = region.bbox

    # Convert normalized coordinates to image pixels
    x1 = x1 / 1000 * width
    x2 = x2 / 1000 * width
    y1 = y1 / 1000 * height
    y2 = y2 / 1000 * height

    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    ax.add_patch(rect)

ax.set_title(f"Gemini Annotation: {case['case_id']}")
ax.axis("off")

plt.show()

Visualize the Gemini-predicted bounding boxes on the ultrasound image